<a href="https://colab.research.google.com/github/nahmeddn-sys/GENAI/blob/main/Assignment_1_Implement_Fine_Tuning_using_an_LLM_(Mistral)_with_a_Small_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Introduction**

In this assignment, a pre-trained LLM is fine-tuned using a medical question–answer dataset consisting of more than 100 examples. The goal is to teach the model to provide more accurate and relevant answers to medical-related queries. The implementation is performed using Google Colab with a free GPU, making it accessible without requiring expensive hardware. By applying modern techniques such as Low Rank Adaptation (LoRA), the model can be fine-tuned efficiently without modifying the entire set of model parameters.

**Explanation of the Approach**

The fine-tuning process begins by selecting a widely used open-source LLM such as Mistral 7B. Instead of training the model from scratch, which would require massive computational resources, the existing pre-trained knowledge of the model is reused. A small but meaningful dataset of medical questions and answers is prepared and formatted into an instruction-response structure. This structured format helps the model learn how to interpret user queries and generate appropriate responses.

To make the training process efficient on limited hardware like the free GPU available in Google Colab, a technique called Low Rank Adaptation (LoRA) is used. LoRA works by keeping the original model weights frozen and introducing small additional layers that learn task-specific knowledge. This significantly reduces memory usage and training time while still improving the model’s performance for the target task.

During training, the model learns to predict the correct response for each instruction in the dataset. After several training iterations, the model becomes better at answering similar medical questions. The final result is a fine-tuned LLM that retains the general language understanding of the original model but performs better for domain-specific queries.

**Install Required Libraries**

In [ ]:
!pip install unsloth
!pip install transformers datasets accelerate peft trl bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Import Libraries**

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


**Load Mistral Model**

We load a quantized version to fit into Colab GPU.

-4bit quantization reduces memory
-Allows 7B model to run on Colab GPU

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.3.3: Fast Mistral patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

**Add LoRA Adapters**



In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
)

Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


**Create Training Dataset**

Example dataset for medical Q&A.

In [ ]:
data = [
{"instruction":"What are the symptoms of diabetes?","output":"Common symptoms include frequent urination, increased thirst, fatigue and blurred vision."},
{"instruction":"What causes high blood pressure?","output":"High blood pressure may be caused by genetics, obesity, stress, smoking and high salt intake."},
{"instruction":"What are the symptoms of dehydration?","output":"Symptoms include dry mouth, dizziness, dark urine and fatigue."},
{"instruction":"How can fever be treated at home?","output":"Drink plenty of fluids, rest and take paracetamol if necessary."},
{"instruction":"What are the symptoms of anemia?","output":"Symptoms include fatigue, weakness, pale skin and shortness of breath."},
{"instruction":"What is a migraine?","output":"A migraine is a severe headache often accompanied by nausea, sensitivity to light and vomiting."},
{"instruction":"What are the symptoms of asthma?","output":"Symptoms include shortness of breath, wheezing, chest tightness and coughing."},
{"instruction":"How can common cold be treated?","output":"Rest, hydration and over-the-counter medications can help relieve symptoms."},
{"instruction":"What causes obesity?","output":"Obesity is caused by excessive calorie intake, lack of physical activity and genetic factors."},
{"instruction":"What are the symptoms of food poisoning?","output":"Symptoms include nausea, vomiting, diarrhea, stomach cramps and fever."},

{"instruction":"What is hypertension?","output":"Hypertension is a medical condition where blood pressure remains consistently high."},
{"instruction":"What are symptoms of flu?","output":"Symptoms include fever, cough, sore throat, body aches and fatigue."},
{"instruction":"How can stress affect health?","output":"Chronic stress can lead to high blood pressure, sleep problems and weakened immunity."},
{"instruction":"What causes insomnia?","output":"Insomnia can be caused by stress, anxiety, irregular sleep habits and certain medications."},
{"instruction":"What are symptoms of vitamin D deficiency?","output":"Symptoms include bone pain, muscle weakness and fatigue."},
{"instruction":"What is dehydration?","output":"Dehydration occurs when the body loses more fluids than it takes in."},
{"instruction":"What are symptoms of appendicitis?","output":"Symptoms include severe abdominal pain, nausea, vomiting and fever."},
{"instruction":"What causes headaches?","output":"Headaches may be caused by stress, dehydration, lack of sleep or eye strain."},
{"instruction":"What are symptoms of pneumonia?","output":"Symptoms include chest pain, cough with phlegm, fever and difficulty breathing."},
{"instruction":"What causes heartburn?","output":"Heartburn occurs when stomach acid flows back into the esophagus."},

{"instruction":"What are symptoms of anxiety?","output":"Symptoms include nervousness, rapid heartbeat, sweating and difficulty concentrating."},
{"instruction":"What is cholesterol?","output":"Cholesterol is a fatty substance found in blood that is needed to build cells."},
{"instruction":"What are symptoms of kidney stones?","output":"Symptoms include severe back pain, blood in urine and nausea."},
{"instruction":"What causes back pain?","output":"Back pain may be caused by muscle strain, poor posture or injury."},
{"instruction":"What are symptoms of allergies?","output":"Symptoms include sneezing, runny nose, itchy eyes and skin rashes."},

{"instruction":"How can immunity be improved?","output":"Healthy diet, regular exercise, adequate sleep and hydration improve immunity."},
{"instruction":"What are symptoms of COVID-19?","output":"Symptoms include fever, cough, fatigue and loss of taste or smell."},
{"instruction":"What causes fatigue?","output":"Fatigue can result from lack of sleep, stress, illness or poor nutrition."},
{"instruction":"What are symptoms of gastritis?","output":"Symptoms include stomach pain, nausea, vomiting and bloating."},
{"instruction":"What causes constipation?","output":"Constipation may occur due to low fiber diet, dehydration or lack of physical activity."},

{"instruction":"What are symptoms of bronchitis?","output":"Symptoms include persistent cough, mucus production and chest discomfort."},
{"instruction":"What causes diarrhea?","output":"Diarrhea can be caused by infections, contaminated food or digestive disorders."},
{"instruction":"What are symptoms of dengue?","output":"Symptoms include high fever, severe headache, joint pain and skin rash."},
{"instruction":"What causes anemia?","output":"Anemia is commonly caused by iron deficiency or blood loss."},
{"instruction":"What are symptoms of malaria?","output":"Symptoms include fever, chills, sweating and headaches."},

{"instruction":"How can dehydration be prevented?","output":"Drink adequate water and avoid excessive heat exposure."},
{"instruction":"What are symptoms of thyroid disorder?","output":"Symptoms may include weight changes, fatigue and mood swings."},
{"instruction":"What causes acne?","output":"Acne is caused by clogged pores, excess oil production and bacteria."},
{"instruction":"What are symptoms of liver disease?","output":"Symptoms include jaundice, abdominal swelling and fatigue."},
{"instruction":"What causes muscle cramps?","output":"Muscle cramps may occur due to dehydration or electrolyte imbalance."}
]
dataset = Dataset.from_list(data)

**Format Dataset**

LLMs expect data in instruction format.

In [ ]:
def format_prompt(example):
    return {
        "text": f"""
### Instruction:
{example['instruction']}

### Response:
{example['output']}
"""
    }

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

**Training Configuration**

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 1,
    output_dir = "outputs",
)

**Train the Model**

In [ ]:
model.save_pretrained("medical_mistral_model")
tokenizer.save_pretrained("medical_mistral_model")

('medical_mistral_model/tokenizer_config.json',
 'medical_mistral_model/chat_template.jinja',
 'medical_mistral_model/tokenizer.json')

**Save Fine Tuned Model**

In [ ]:
model.save_pretrained("medical_mistral_model")
tokenizer.save_pretrained("medical_mistral_model")

('medical_mistral_model/tokenizer_config.json',
 'medical_mistral_model/chat_template.jinja',
 'medical_mistral_model/tokenizer.json')

**Test the Model**

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
"""### Instruction:
What are the symptoms of diabetes?

### Response:
"""
], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0]))

<s> ### Instruction:
What are the symptoms of diabetes?

### Response:

Diabetes, specifically type 1 and type 2, can present with various symptoms. Here are some common symptoms:

1. Increased thirst and frequent urination: The body tries to get rid of the excess sugar in the blood by excreting it in the urine, leading to increased urination and dehydration.

2. Increased hunger: The body may not be able to effectively use the sugar in the blood for energy, leading to
